In [ ]:
# SINGLE CONFIGURATION BLOCK — edit only this cell before a run.
RUN_MODE = "smoke"  # smoke or full
OUTPUT_DIR = "/content/counterfactual_faithfulness_stage2c"
SEED = 53
MODEL_NAME = ["dino_wm_pusht", "jepa_wm_pusht"]
ENVIRONMENT = "PushT"
HORIZONS = [3, 6]
NUM_STATES = 18
ACTIONS_PER_STATE = 10

MOUNT_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage2c"
REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
FRAMESKIP = 5
FEATURE_POOL_GRID = 4
CANDIDATE_LIBRARY_SIZE = 22
PROBE_SPLIT = [0.50, 0.20, 0.30]  # train, calibration, final test
READOUT_PROJECTION_DIM = 256
RIDGE_LAMBDAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
MLP_HIDDEN = 128
MLP_MAX_EPOCHS = 20
MLP_PATIENCE = 5
MLP_LEARNING_RATE = 1e-3
MLP_WEIGHT_DECAY = 1e-4
BOOTSTRAP_REPS = 500
RANKING_TIE = 1e-9

if RUN_MODE == "full":
    NUM_STATES = 300
    MLP_MAX_EPOCHS = 200
    MLP_PATIENCE = 20
    BOOTSTRAP_REPS = 2000
elif RUN_MODE != "smoke":
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")

assert MODEL_NAME == ["dino_wm_pusht", "jepa_wm_pusht"]
assert ENVIRONMENT == "PushT"
assert HORIZONS == [3, 6]
assert 8 <= ACTIONS_PER_STATE <= 12
assert abs(sum(PROBE_SPLIT) - 1.0) < 1e-12


# Stage 2C: task-aligned readout diagnosis

Stage 2B established two facts at once: the frozen world models predict
action-specific latent changes, yet aggregate latent counterfactual error
does not add held-out information about executable physical regret beyond
ordinary rollout error and difficulty covariates.

The suspected mismatch is concrete. The existing planner ranks actions by
Euclidean distance between a predicted latent and a single goal-image
latent. Stage 2C tests whether the actionable physical state is present but
needs a task-aligned readout.

The world models remain frozen. For each checkpoint, a state-disjoint probe
decodes `(block_x, block_y, sin(theta), cos(theta))` from predicted future
latents. A linear ridge probe is primary; a small MLP is secondary. Probe
hyperparameters are chosen on calibration states, and all planning metrics
are reported only on untouched test states.

The action set is the same fixed state-relative set used in Stage 2B and is
never selected using future simulator outcomes. Horizons 3 and 6 are used;
horizon 1 is dropped because its no-op oracle rate remained high.

Primary gate: relative to raw latent-goal distance, a task-aligned readout
must improve both physical normalized regret and margin-weighted pair
ranking with state-clustered 95% bootstrap intervals entirely above zero.
A linear-probe pass is `TASK_ALIGNED_SIGNAL`; an MLP-only pass is
`NONLINEAR_TASK_ALIGNED_SIGNAL`.

Run all cells from a fresh GPU runtime. A 16 GB T4 is sufficient; an L4 or
A100 is recommended. Google Drive is optional. Per-state simulator and model
shards are resumable. `stage2c_result_bundle.zip` downloads automatically
after success or a captured failure.


In [ ]:
import subprocess
import sys

# Keep Colab's CUDA-matched torch and torchvision builds.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")
print("No runtime restart is expected.")


In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import yaml

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

OUT = Path(OUTPUT_DIR)
INTERMEDIATE = OUT / "intermediate"
TRUTH_DIR = INTERMEDIATE / "truth"
MODEL_ROOT = INTERMEDIATE / "models"
LOG_DIR = OUT / "logs"
PLOT_DIR = OUT / "plots"
PROBE_DIR = OUT / "probes"
for path in [OUT, INTERMEDIATE, TRUTH_DIR, MODEL_ROOT, LOG_DIR, PLOT_DIR, PROBE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_bytes": torch.cuda.get_device_properties(0).total_memory,
}
print(json.dumps(VERSIONS, indent=2))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage2c")
log.info("Seeds set to %d", SEED)

def gpu_report(label):
    payload = {
        "label": label,
        "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
        "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3),
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3),
    }
    log.info("GPU memory %s", payload)
    return payload

CONFIG = {
    "RUN_MODE": RUN_MODE,
    "OUTPUT_DIR": OUTPUT_DIR,
    "SEED": SEED,
    "MODEL_NAME": MODEL_NAME,
    "ENVIRONMENT": ENVIRONMENT,
    "HORIZONS": HORIZONS,
    "NUM_STATES": NUM_STATES,
    "ACTIONS_PER_STATE": ACTIONS_PER_STATE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "REPO_URL": REPO_URL,
    "REPO_COMMIT": REPO_COMMIT,
    "FRAMESKIP": FRAMESKIP,
    "FEATURE_POOL_GRID": FEATURE_POOL_GRID,
    "BOOTSTRAP_REPS": BOOTSTRAP_REPS,
    "CANDIDATE_LIBRARY_SIZE": CANDIDATE_LIBRARY_SIZE,
    "PROBE_SPLIT": PROBE_SPLIT,
    "READOUT_PROJECTION_DIM": READOUT_PROJECTION_DIM,
    "RIDGE_LAMBDAS": RIDGE_LAMBDAS,
    "MLP_HIDDEN": MLP_HIDDEN,
    "MLP_MAX_EPOCHS": MLP_MAX_EPOCHS,
    "MLP_PATIENCE": MLP_PATIENCE,
    "MLP_LEARNING_RATE": MLP_LEARNING_RATE,
    "MLP_WEIGHT_DECAY": MLP_WEIGHT_DECAY,
    "RANKING_TIE": RANKING_TIE,
    "pinned_dependencies": PINNED,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()
CONFIG["run_signature"] = RUN_SIGNATURE
config_path = OUT / "config.json"
if config_path.exists():
    previous = json.loads(config_path.read_text())
    if previous.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "OUTPUT_DIR contains a different configuration; choose a new OUTPUT_DIR."
        )
config_path.write_text(json.dumps(CONFIG, indent=2) + "\n")
(OUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

PIPELINE_FAILED = False
FAILURE_MESSAGE = ""

def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)

gpu_report("startup")


In [ ]:
# Core simulator, metric, model-loading, and statistical helpers.
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value

def write_json(path, payload):
    Path(path).write_text(json.dumps(json_ready(payload), indent=2) + "\n")

def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        raise ValueError(f"no rows for {path}")
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_suffix(".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def pool_visual(visual, grid=FEATURE_POOL_GRID):
    value = visual.detach().float().cpu().numpy()[..., 0, :, :, :]
    height, width, dim = value.shape[-3:]
    if height % grid or width % grid:
        raise ValueError(f"feature grid {(height, width)} is not divisible by {grid}")
    fh, fw = height // grid, width // grid
    value = value.reshape(*value.shape[:-3], grid, fh, grid, fw, dim)
    value = value.mean(axis=(-4, -2))
    return value.reshape(*value.shape[:-3], -1)

def pair_indices(n_actions):
    left, right = [], []
    for i in range(n_actions):
        for j in range(i + 1, n_actions):
            left.append(i)
            right.append(j)
    return np.asarray(left), np.asarray(right)

def counterfactual_arrays(truth, prediction, eps=1e-12):
    truth = np.asarray(truth, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    if truth.shape != prediction.shape or truth.ndim != 3:
        raise ValueError(f"expected matching [action,horizon,feature], got {truth.shape}")
    errors = prediction - truth
    ordinary = np.sqrt(np.mean(errors**2, axis=(0, 2)))
    common = np.mean(errors, axis=0)
    common_rmse = np.sqrt(np.mean(common**2, axis=-1))
    centered = errors - common[None, :, :]
    action_dependent = np.sqrt(np.mean(centered**2, axis=(0, 2)))
    left, right = pair_indices(truth.shape[0])
    dy = truth[left] - truth[right]
    dy_hat = prediction[left] - prediction[right]
    pair_error = dy_hat - dy
    pair_rmse = np.sqrt(np.mean(pair_error**2, axis=-1))
    pair_scale = np.sqrt(np.mean(dy**2, axis=-1))
    pair_normalized = pair_rmse / np.maximum(pair_scale, eps)
    dot = np.sum(dy_hat * dy, axis=-1)
    denom = np.linalg.norm(dy_hat, axis=-1) * np.linalg.norm(dy, axis=-1)
    # A zero predicted intervention receives zero directional alignment.
    pair_cosine = np.divide(
        dot, denom, out=np.zeros_like(dot), where=denom > eps
    )
    aggregate_pair_rmse = np.sqrt(np.mean(pair_error**2, axis=(0, 2)))
    aggregate_scale = np.sqrt(np.mean(dy**2, axis=(0, 2)))
    aggregate_normalized = aggregate_pair_rmse / np.maximum(aggregate_scale, eps)
    aggregate_cosine = np.mean(pair_cosine, axis=0)
    expected_pair_mse = (
        2 * truth.shape[0] / (truth.shape[0] - 1)
    ) * action_dependent**2
    identity_residual = aggregate_pair_rmse**2 - expected_pair_mse
    return {
        "ordinary_rmse": ordinary,
        "common_mode_rmse": common_rmse,
        "action_dependent_rmse": action_dependent,
        "paired_effect_rmse": aggregate_pair_rmse,
        "ground_truth_effect_rms": aggregate_scale,
        "normalized_paired_effect_rmse": aggregate_normalized,
        "paired_effect_cosine": aggregate_cosine,
        "identity_residual": identity_residual,
        "pair_left": left,
        "pair_right": right,
        "pair_effect_rmse": pair_rmse,
        "pair_effect_scale": pair_scale,
        "pair_normalized_effect_rmse": pair_normalized,
        "pair_effect_cosine": pair_cosine,
    }

def ranking_arrays(true_cost, predicted_cost, eps=1e-12, tie=1e-9):
    true_cost = np.asarray(true_cost, dtype=np.float64)
    predicted_cost = np.asarray(predicted_cost, dtype=np.float64)
    if true_cost.shape != predicted_cost.shape or true_cost.ndim != 2:
        raise ValueError("costs must match with shape [action,horizon]")
    selected = np.argmin(predicted_cost, axis=0)
    oracle = np.argmin(true_cost, axis=0)
    horizon_index = np.arange(true_cost.shape[1])
    chosen = true_cost[selected, horizon_index]
    best = np.min(true_cost, axis=0)
    regret = chosen - best
    spread = np.max(true_cost, axis=0) - best
    normalized_regret = np.divide(
        regret,
        np.maximum(spread, eps),
        out=np.zeros_like(regret),
        where=spread > eps,
    )
    top1 = (chosen <= best + tie).astype(np.float64)
    left, right = pair_indices(true_cost.shape[0])
    true_delta = true_cost[left] - true_cost[right]
    pred_delta = predicted_cost[left] - predicted_cost[right]
    valid = np.abs(true_delta) > tie
    pair_credit = np.full_like(true_delta, np.nan)
    pair_credit[valid & (np.sign(true_delta) == np.sign(pred_delta))] = 1.0
    pair_credit[valid & (np.abs(pred_delta) <= tie)] = 0.5
    pair_credit[valid & np.isnan(pair_credit)] = 0.0
    pairwise_accuracy = np.nanmean(pair_credit, axis=0)
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": top1,
        "regret": regret,
        "normalized_regret": normalized_regret,
        "pairwise_accuracy": pairwise_accuracy,
        "pairwise_credit": pair_credit,
    }

def task_cost(states, goal=np.array([256.0, 256.0, np.pi / 4])):
    states = np.asarray(states)
    angle = np.arctan2(
        np.sin(states[..., 4] - goal[2]),
        np.cos(states[..., 4] - goal[2]),
    )
    pieces = np.concatenate(
        [(states[..., 2:4] - goal[:2]) / 512.0, (angle / np.pi)[..., None]],
        axis=-1,
    )
    return np.linalg.norm(pieces, axis=-1)

def unit_vector(vector):
    vector = np.asarray(vector, dtype=np.float64)
    norm = np.linalg.norm(vector)
    if norm < 1e-12:
        raise ValueError("zero direction")
    return vector / norm

def rotate_vector(vector, degrees):
    radians = np.deg2rad(degrees)
    matrix = np.array(
        [
            [np.cos(radians), -np.sin(radians)],
            [np.sin(radians), np.cos(radians)],
        ]
    )
    return matrix @ vector

def build_states(count):
    rng = np.random.default_rng(SEED)
    states, strata = [], []
    goal_xy = np.array([256.0, 256.0])
    for index in range(count):
        radial_distance = rng.uniform(90.0, 130.0)
        polar_angle = rng.uniform(-np.pi, np.pi)
        block = goal_xy + radial_distance * np.array(
            [np.cos(polar_angle), np.sin(polar_angle)]
        )
        push_direction = unit_vector(goal_xy - block)
        if index % 2 == 0:
            agent_distance = rng.uniform(60.0, 68.0)
            stratum = "near"
        else:
            agent_distance = rng.uniform(72.0, 80.0)
            stratum = "far"
        agent = block - agent_distance * push_direction
        if np.any(agent < 35.0) or np.any(agent > 477.0):
            raise AssertionError(f"agent out of bounds: {agent}")
        block_angle = rng.uniform(-0.60, 0.60)
        states.append(
            [agent[0], agent[1], block[0], block[1], block_angle, 0.0, 0.0]
        )
        strata.append(stratum)
    return np.asarray(states, dtype=np.float64), np.asarray(strata)

def candidate_library(state, primitive_steps):
    goal_xy = np.array([256.0, 256.0])
    push_direction = unit_vector(goal_xy - np.asarray(state)[2:4])
    specifications = [("noop", 0.0, 0)]
    specifications.extend(
        (f"direct_{duration}", 0.0, duration)
        for duration in [8, 12, 16, 20, 24, 30]
    )
    for angle in [-20.0, 20.0]:
        for duration in [12, 18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-40.0, 40.0]:
        for duration in [18, 24]:
            specifications.append(
                (f"angle_{angle:+.0f}_{duration}", angle, duration)
            )
    for angle in [-70.0, 70.0, 140.0, -140.0, 180.0]:
        specifications.append((f"angle_{angle:+.0f}_24", angle, 24))
    if len(specifications) != CANDIDATE_LIBRARY_SIZE:
        raise AssertionError("candidate-library size changed")

    sequences = []
    for _, angle, duration in specifications:
        sequence = np.zeros((primitive_steps, 2), dtype=np.float32)
        if duration:
            sequence[:duration] = (
                0.14 * rotate_vector(push_direction, angle)
            ).astype(np.float32)
        sequences.append(sequence)
    return np.stack(sequences), [item[0] for item in specifications]

def fixed_candidate_indices():
    # Frozen before confirmatory model evaluation. These indices cover
    # no-op, multiple push durations, and symmetric angular deviations.
    chosen = np.asarray([0, 2, 4, 6, 8, 11, 14, 16, 17, 18], dtype=np.int64)
    if len(chosen) != ACTIONS_PER_STATE or len(np.unique(chosen)) != len(chosen):
        raise AssertionError("invalid fixed candidate subset")
    if np.min(chosen) < 0 or np.max(chosen) >= CANDIDATE_LIBRARY_SIZE:
        raise AssertionError("fixed candidate index outside library")
    return chosen

def configure_repo():
    repo = CACHE_ROOT / "jepa-wms"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT], check=True)
    resolved = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in Push-T predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    configs = [
        repo / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in configs:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo

def make_environment(repo):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv
    return PushTEnv(
        with_velocity=True,
        with_target=True,
        render_size=224,
        relative=True,
        action_scale=100,
    )

def reset_env(env, state, seed):
    env.seed(seed)
    env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
    observation, restored = env.reset()
    return {
        "visual": np.asarray(observation["visual"]).copy(),
        "proprio": np.asarray(observation["proprio"]).copy(),
    }, np.asarray(restored).copy()

def rollout_branch(env, state, primitive_actions, horizons, seed):
    observation0, restored = reset_env(env, state, seed)
    wanted = set(horizons)
    observations, states, contacts, coverages = {}, {}, {}, {}
    cumulative_contacts = 0
    for step, action in enumerate(primitive_actions, start=1):
        observation, _, _, info = env.step(action)
        cumulative_contacts += int(info.get("n_contacts", 0))
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = {
                    "visual": np.asarray(observation["visual"]).copy(),
                    "proprio": np.asarray(observation["proprio"]).copy(),
                }
                states[horizon] = np.asarray(info["state"]).copy()
                contacts[horizon] = cumulative_contacts
                coverages[horizon] = float(info["final_coverage"])
    if wanted != set(observations):
        raise RuntimeError(f"missing simulator horizons: {wanted - set(observations)}")
    return observation0, restored, observations, states, contacts, coverages

def exact_restore_test(env, state, actions, repeats=3):
    endpoints, images, diagnostics = [], [], []
    for _ in range(repeats):
        initial, _, _, states, contacts, coverages = rollout_branch(
            env, state, actions, [max(HORIZONS)], SEED + 5000
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        diagnostics.append((contacts[max(HORIZONS)], coverages[max(HORIZONS)]))
    result = {
        "repeats": repeats,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            diagnostics[0] == item for item in diagnostics[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(np.max(np.abs(endpoints[0] - item)) for item in endpoints[1:])
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result

def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim == 5:
        pass
    else:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}

def state_metrics(truth_features, prediction_features, physical_cost, coverage_cost, contacts):
    variants = {
        "real": prediction_features,
        "action_blind": np.broadcast_to(
            np.mean(prediction_features, axis=0, keepdims=True),
            prediction_features.shape,
        ).copy(),
        "action_shuffled": prediction_features[
            np.roll(np.arange(prediction_features.shape[0]), 1)
        ],
    }
    output = {"variant_names": np.asarray(list(variants))}
    latent_true_cost = np.sqrt(
        np.mean((truth_features - CURRENT_GOAL_FEATURE[None, None, :]) ** 2, axis=-1)
    )
    output["latent_true_cost"] = latent_true_cost
    output["physical_true_cost"] = physical_cost
    output["coverage_true_cost"] = coverage_cost
    output["contacts"] = contacts
    metric_names = None
    stored = {}
    for variant_name, predicted in variants.items():
        metrics = counterfactual_arrays(truth_features, predicted)
        if metric_names is None:
            metric_names = list(metrics)
        predicted_latent_cost = np.sqrt(
            np.mean(
                (predicted - CURRENT_GOAL_FEATURE[None, None, :]) ** 2,
                axis=-1,
            )
        )
        rank_latent = ranking_arrays(latent_true_cost, predicted_latent_cost)
        rank_physical = ranking_arrays(physical_cost, predicted_latent_cost)
        rank_coverage = ranking_arrays(coverage_cost, predicted_latent_cost)
        stored[variant_name] = {
            **metrics,
            "latent_predicted_cost": predicted_latent_cost,
            **{f"latent_{key}": value for key, value in rank_latent.items()},
            **{f"physical_{key}": value for key, value in rank_physical.items()},
            **{f"coverage_{key}": value for key, value in rank_coverage.items()},
        }
    for key in stored["real"]:
        output[key] = np.stack([stored[name][key] for name in variants])
    return output


In [ ]:
# Phase A — execute a fixed, state-relative candidate set in the simulator.
def generate_simulator_truth():
    repo = configure_repo()
    env = make_environment(repo)
    states, state_strata = build_states(NUM_STATES)
    primitive_steps = max(HORIZONS) * FRAMESKIP
    goal_state = np.array(
        [80.0, 450.0, 256.0, 256.0, np.pi / 4, 0.0, 0.0]
    )
    goal_observation, _ = reset_env(env, goal_state, SEED + 7000)

    first_library, library_labels = candidate_library(states[0], primitive_steps)
    fixed_indices = fixed_candidate_indices()
    restore = exact_restore_test(
        env, states[0], first_library[fixed_indices[1]], repeats=3
    )
    write_json(OUT / "restore_test.json", restore)
    log.info("Exact restoration: %s", restore)

    for state_index, state in enumerate(states):
        state_path = TRUTH_DIR / f"state_{state_index:04d}.npz"
        if state_path.exists():
            log.info("Simulator resume: keeping %s", state_path.name)
            continue
        library, labels = candidate_library(state, primitive_steps)
        if labels != library_labels:
            raise AssertionError("candidate labels vary by state")
        selected = fixed_candidate_indices()
        visuals, proprios, endpoints = [], [], []
        contacts, coverages, initials = [], [], []
        for actions in library[selected]:
            initial, restored, observations, states_by_h, contacts_by_h, coverage_by_h = (
                rollout_branch(
                    env,
                    state,
                    actions,
                    HORIZONS,
                    SEED + state_index,
                )
            )
            initials.append(initial["visual"])
            visuals.append([observations[h]["visual"] for h in HORIZONS])
            proprios.append([observations[h]["proprio"] for h in HORIZONS])
            endpoints.append([states_by_h[h] for h in HORIZONS])
            contacts.append([contacts_by_h[h] for h in HORIZONS])
            coverages.append([coverage_by_h[h] for h in HORIZONS])
        if not all(np.array_equal(initials[0], item) for item in initials[1:]):
            raise AssertionError(
                f"branch initial render mismatch at state {state_index}"
            )

        endpoint_array = np.asarray(endpoints)
        contact_array = np.asarray(contacts, dtype=np.int32)
        physical_cost = task_cost(endpoint_array)
        atomic_npz(
            state_path,
            initial_state=state,
            design_stratum=state_strata[state_index],
            initial_visual=initials[0],
            initial_proprio=reset_env(
                env, state, SEED + state_index
            )[0]["proprio"],
            selected_actions=library[selected],
            selected_library_indices=selected,
            future_visual=np.asarray(visuals, dtype=np.uint8),
            future_proprio=np.asarray(proprios, dtype=np.float32),
            endpoint_states=endpoint_array.astype(np.float32),
            contacts=contact_array,
            coverage=np.asarray(coverages, dtype=np.float32),
            physical_cost=physical_cost.astype(np.float32),
        )
        write_json(
            OUT / "simulator_progress.json",
            {
                "run_signature": RUN_SIGNATURE,
                "completed_states": state_index + 1,
                "total_states": NUM_STATES,
                "last_file": state_path.name,
            },
        )
        log.info("Simulator state %d/%d", state_index + 1, NUM_STATES)

    action_bank, selected_indices = [], []
    all_costs, all_contacts = [], []
    for state_index in range(NUM_STATES):
        with np.load(TRUTH_DIR / f"state_{state_index:04d}.npz") as shard:
            action_bank.append(shard["selected_actions"])
            selected_indices.append(shard["selected_library_indices"])
            all_costs.append(shard["physical_cost"])
            all_contacts.append(shard["contacts"])
    action_bank = np.asarray(action_bank, dtype=np.float32)
    selected_indices = np.asarray(selected_indices, dtype=np.int64)
    all_costs = np.asarray(all_costs, dtype=np.float64)
    all_contacts = np.asarray(all_contacts, dtype=np.int32)

    oracle = np.argmin(all_costs, axis=1)
    spread = np.max(all_costs, axis=1) - np.min(all_costs, axis=1)
    no_op_regret = all_costs[:, 0] - np.min(all_costs, axis=1)
    left, right = pair_indices(ACTIONS_PER_STATE)
    pair_contact_count = (
        (all_contacts[:, left, :] > 0).astype(int)
        + (all_contacts[:, right, :] > 0).astype(int)
    )
    design_summary = {
        "candidate_library_size": CANDIDATE_LIBRARY_SIZE,
        "selected_actions_per_state": ACTIONS_PER_STATE,
        "selection_protocol": (
            "fixed state-relative subset frozen before model evaluation; "
            "future simulator outcomes are not used for candidate selection"
        ),
        "no_op_oracle_fraction_by_horizon": np.mean(
            oracle == 0, axis=0
        ).tolist(),
        "no_op_positive_regret_fraction_by_horizon": np.mean(
            no_op_regret > 1e-9, axis=0
        ).tolist(),
        "median_physical_cost_spread_by_horizon": np.median(
            spread, axis=0
        ).tolist(),
        "minimum_physical_cost_spread_by_horizon": np.min(
            spread, axis=0
        ).tolist(),
        "contact_fraction_by_horizon": np.mean(
            all_contacts > 0, axis=(0, 1)
        ).tolist(),
        "pair_contact_counts": {
            label: int(np.sum(pair_contact_count == index))
            for index, label in enumerate(["neither", "one", "both"])
        },
    }
    if design_summary["no_op_oracle_fraction_by_horizon"][-1] >= 0.20:
        raise AssertionError("final-horizon no-op oracle remains degenerate")
    if design_summary["no_op_positive_regret_fraction_by_horizon"][-1] <= 0.80:
        raise AssertionError("final-horizon no-op regret is insufficient")
    if design_summary["median_physical_cost_spread_by_horizon"][-1] <= 0.08:
        raise AssertionError("final-horizon physical cost spread is insufficient")
    if not all(
        design_summary["pair_contact_counts"][label] > 0
        for label in ["neither", "one", "both"]
    ):
        raise AssertionError("candidate design is missing a contact stratum")
    write_json(OUT / "candidate_design_summary.json", design_summary)

    atomic_npz(
        OUT / "design.npz",
        states=states,
        state_strata=state_strata,
        action_bank=action_bank,
        selected_library_indices=selected_indices,
        candidate_library_labels=np.asarray(library_labels),
        goal_state=goal_state,
        goal_visual=goal_observation["visual"],
        goal_proprio=goal_observation["proprio"],
    )

    import pymunk
    write_json(
        OUT / "environment.json",
        {
            "name": "JEPA-WMs bundled PushTEnv",
            "repository": REPO_URL,
            "commit": REPO_COMMIT,
            "pymunk": pymunk.version,
            "relative_actions": True,
            "action_scale": 100,
            "with_velocity": True,
            "frameskip": FRAMESKIP,
            "branch_protocol": "fresh simulator space per action branch",
            "candidate_protocol": "fixed state-relative action subset",
            "action_pair_contact_strata": ["neither", "one", "both"],
        },
    )
    return repo

if not PIPELINE_FAILED:
    try:
        REPO = generate_simulator_truth()
    except Exception:
        record_failure("simulator_truth")


In [ ]:
# Phase B — evaluate frozen checkpoints and retain compact probe features.
def evaluate_models():
    global CURRENT_GOAL_FEATURE
    repo = configure_repo()
    with np.load(OUT / "design.npz") as design:
        action_bank = design["action_bank"]
        goal_visual = design["goal_visual"]
        goal_proprio = design["goal_proprio"]
    max_horizon = max(HORIZONS)

    for model_index, model_name in enumerate(MODEL_NAME):
        model_dir = MODEL_ROOT / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        torch.cuda.reset_peak_memory_stats()
        gpu_report(f"{model_name}_before_load")
        model, preprocessor = torch.hub.load(
            str(repo),
            model_name,
            source="local",
            pretrained=True,
            device="cuda:0",
            trust_repo=True,
        )
        model.eval()
        gpu_report(f"{model_name}_after_load")
        with torch.inference_mode():
            goal_encoded = model.encode(
                to_model_observation(goal_visual, goal_proprio)
            )
            CURRENT_GOAL_FEATURE = pool_visual(goal_encoded["visual"])[0, 0]

        horizon_index = torch.tensor(HORIZONS, dtype=torch.long, device="cuda")

        for state_index in range(NUM_STATES):
            output_path = model_dir / f"state_{state_index:04d}.npz"
            if output_path.exists():
                log.info("%s resume: keeping %s", model_name, output_path.name)
                continue
            with np.load(TRUTH_DIR / f"state_{state_index:04d}.npz") as truth_shard:
                initial_visual = truth_shard["initial_visual"]
                initial_proprio = truth_shard["initial_proprio"]
                future_visual = truth_shard["future_visual"]
                future_proprio = truth_shard["future_proprio"]
                physical_cost = truth_shard["physical_cost"].astype(np.float64)
                coverage_cost = 1.0 - truth_shard["coverage"].astype(np.float64)
                contacts = truth_shard["contacts"]
            chunks = torch.from_numpy(
                action_bank[state_index].reshape(
                    ACTIONS_PER_STATE, max_horizon, FRAMESKIP, 2
                )
            ).float()
            normalized_chunks = preprocessor.normalize_actions(chunks)
            model_actions = (
                normalized_chunks.reshape(ACTIONS_PER_STATE, max_horizon, -1)
                .permute(1, 0, 2)
                .contiguous()
                .cuda()
            )
            with torch.inference_mode():
                initial_encoded = model.encode(
                    to_model_observation(initial_visual, initial_proprio)
                )
                truth_encoded = model.encode(
                    to_model_observation(future_visual, future_proprio)
                )
                truth_features = pool_visual(truth_encoded["visual"])
                predicted_encoded = model.unroll(initial_encoded, model_actions)
                predicted_selected = predicted_encoded["visual"].index_select(
                    0, horizon_index
                )
                predicted_features = np.moveaxis(
                    pool_visual(predicted_selected), 0, 1
                )
            metrics = state_metrics(
                truth_features,
                predicted_features,
                physical_cost,
                coverage_cost,
                contacts,
            )
            atomic_npz(
                output_path,
                **metrics,
                predicted_features=predicted_features.astype(np.float16),
            )
            write_json(
                OUT / f"{model_name}_progress.json",
                {
                    "run_signature": RUN_SIGNATURE,
                    "model": model_name,
                    "completed_states": state_index + 1,
                    "total_states": NUM_STATES,
                    "last_file": output_path.name,
                },
            )
            log.info("%s state %d/%d", model_name, state_index + 1, NUM_STATES)
            if (state_index + 1) % 25 == 0 or RUN_MODE == "smoke":
                gpu_report(f"{model_name}_state_{state_index:04d}")

        del model, preprocessor, goal_encoded, initial_encoded
        gc.collect()
        torch.cuda.empty_cache()
        gpu_report(f"{model_name}_released")

if not PIPELINE_FAILED:
    try:
        evaluate_models()
    except Exception:
        record_failure("model_evaluation")


In [ ]:
# Phase C — fit task-aligned readouts on state-disjoint splits and evaluate.
from copy import deepcopy

import torch.nn as nn

READOUTS = [
    "latent_distance",
    "linear_pose",
    "mlp_pose",
    "action_blind",
    "linear_pose_shuffled",
    "mlp_pose_shuffled",
    "oracle_pose",
]

def make_state_split(count, fractions, seed):
    if count < 12:
        raise ValueError("at least 12 states are required for a three-way split")
    rng = np.random.default_rng(seed)
    order = rng.permutation(count)
    train_count = max(4, int(np.floor(count * fractions[0])))
    calibration_count = max(3, int(np.floor(count * fractions[1])))
    if train_count + calibration_count > count - 3:
        calibration_count = count - train_count - 3
    train = np.sort(order[:train_count])
    calibration = np.sort(order[train_count : train_count + calibration_count])
    test = np.sort(order[train_count + calibration_count :])
    if min(len(train), len(calibration), len(test)) < 3:
        raise AssertionError("state split is too small")
    combined = np.concatenate([train, calibration, test])
    if len(np.unique(combined)) != count:
        raise AssertionError("state split overlaps or omits states")
    return {
        "train": train,
        "calibration": calibration,
        "test": test,
    }

def pose_target(endpoint_states):
    states = np.asarray(endpoint_states, dtype=np.float64)
    theta = states[..., 4]
    return np.stack(
        [
            states[..., 2] / 512.0,
            states[..., 3] / 512.0,
            np.sin(theta),
            np.cos(theta),
        ],
        axis=-1,
    )

def angle_from_pose_target(target):
    target = np.asarray(target, dtype=np.float64)
    return np.arctan2(target[..., 2], target[..., 3])

def task_cost_from_pose_target(
    target,
    goal=np.array([256.0, 256.0, np.pi / 4]),
):
    target = np.asarray(target, dtype=np.float64)
    xy_error = target[..., :2] - np.asarray(goal[:2]) / 512.0
    angle = angle_from_pose_target(target)
    angle_error = np.arctan2(
        np.sin(angle - goal[2]),
        np.cos(angle - goal[2]),
    )
    pieces = np.concatenate(
        [xy_error, (angle_error / np.pi)[..., None]],
        axis=-1,
    )
    return np.linalg.norm(pieces, axis=-1)

def physical_pose_error(prediction, truth):
    prediction = np.asarray(prediction, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    xy_error = prediction[..., :2] - truth[..., :2]
    angle_error = np.arctan2(
        np.sin(angle_from_pose_target(prediction) - angle_from_pose_target(truth)),
        np.cos(angle_from_pose_target(prediction) - angle_from_pose_target(truth)),
    )
    pieces = np.concatenate(
        [xy_error, (angle_error / np.pi)[..., None]],
        axis=-1,
    )
    return np.linalg.norm(pieces, axis=-1)

def random_projection(input_dim, output_dim, seed):
    if output_dim <= 0 or output_dim > input_dim:
        raise ValueError("invalid projection dimension")
    rng = np.random.default_rng(seed)
    return (
        rng.standard_normal((input_dim, output_dim)).astype(np.float32)
        / np.sqrt(output_dim)
    )

def standardize_fit(x):
    x = np.asarray(x, dtype=np.float64)
    mean = np.mean(x, axis=0)
    scale = np.std(x, axis=0)
    scale[scale < 1e-8] = 1.0
    return mean, scale

def fit_linear_pose(x_train, y_train, x_calibration, y_calibration, lambdas):
    mean, scale = standardize_fit(x_train)
    train = (np.asarray(x_train, dtype=np.float64) - mean) / scale
    calibration = (np.asarray(x_calibration, dtype=np.float64) - mean) / scale
    train = np.column_stack([np.ones(len(train)), train])
    calibration = np.column_stack([np.ones(len(calibration)), calibration])
    gram = train.T @ train
    cross = train.T @ np.asarray(y_train, dtype=np.float64)
    best = None
    for ridge in lambdas:
        penalty = np.eye(gram.shape[0]) * float(ridge)
        penalty[0, 0] = 0.0
        coefficient = np.linalg.solve(gram + penalty, cross)
        prediction = calibration @ coefficient
        loss = float(np.mean((prediction - y_calibration) ** 2))
        candidate = {
            "ridge": float(ridge),
            "calibration_pose_mse": loss,
            "mean": mean,
            "scale": scale,
            "coefficient": coefficient,
        }
        if best is None or candidate["calibration_pose_mse"] < best["calibration_pose_mse"]:
            best = candidate
    return best

def predict_linear_pose(probe, x):
    standardized = (np.asarray(x, dtype=np.float64) - probe["mean"]) / probe["scale"]
    augmented = np.column_stack([np.ones(len(standardized)), standardized])
    return augmented @ probe["coefficient"]

class PoseMLP(nn.Module):
    def __init__(self, input_dim, hidden):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, 4),
        )

    def forward(self, value):
        return self.network(value)

def fit_mlp_pose(x_train, y_train, x_calibration, y_calibration, seed):
    mean, scale = standardize_fit(x_train)
    train_x = torch.from_numpy(
        ((np.asarray(x_train) - mean) / scale).astype(np.float32)
    ).cuda()
    train_y = torch.from_numpy(np.asarray(y_train, dtype=np.float32)).cuda()
    calibration_x = torch.from_numpy(
        ((np.asarray(x_calibration) - mean) / scale).astype(np.float32)
    ).cuda()
    calibration_y = torch.from_numpy(
        np.asarray(y_calibration, dtype=np.float32)
    ).cuda()
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    model = PoseMLP(train_x.shape[1], MLP_HIDDEN).cuda()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=MLP_LEARNING_RATE,
        weight_decay=MLP_WEIGHT_DECAY,
    )
    best_loss = float("inf")
    best_epoch = -1
    best_state = None
    stale = 0
    for epoch in range(MLP_MAX_EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        prediction = model(train_x)
        loss = torch.mean((prediction - train_y) ** 2)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.inference_mode():
            calibration_prediction = model(calibration_x)
            calibration_loss = float(
                torch.mean((calibration_prediction - calibration_y) ** 2).item()
            )
        if calibration_loss < best_loss - 1e-9:
            best_loss = calibration_loss
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            stale = 0
        else:
            stale += 1
        if stale >= MLP_PATIENCE:
            break
    if best_state is None:
        raise AssertionError("MLP probe did not produce a checkpoint")
    model.load_state_dict(best_state)
    model.cpu()
    del train_x, train_y, calibration_x, calibration_y
    torch.cuda.empty_cache()
    return {
        "mean": mean,
        "scale": scale,
        "model": model,
        "best_epoch": int(best_epoch),
        "calibration_pose_mse": float(best_loss),
    }

def predict_mlp_pose(probe, x, batch_size=2048):
    standardized = (
        (np.asarray(x, dtype=np.float64) - probe["mean"]) / probe["scale"]
    ).astype(np.float32)
    model = probe["model"].cuda().eval()
    pieces = []
    with torch.inference_mode():
        for start in range(0, len(standardized), batch_size):
            batch = torch.from_numpy(
                standardized[start : start + batch_size]
            ).cuda()
            pieces.append(model(batch).cpu().numpy())
    model.cpu()
    torch.cuda.empty_cache()
    return np.concatenate(pieces, axis=0)

def ranking_metrics(true_cost, predicted_cost, tie=RANKING_TIE):
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    if truth.shape != prediction.shape or truth.ndim != 1:
        raise ValueError("ranking inputs must be matching action vectors")
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    regret = chosen - best
    normalized_regret = regret / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    pairwise = float(np.nanmean(credit)) if np.any(valid) else float("nan")
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else float("nan")
    )
    margin_scale = float(np.sqrt(np.mean(true_margin[valid] ** 2))) if np.any(valid) else 0.0
    normalized_margin_rmse = (
        float(
            np.sqrt(np.mean((predicted_margin[valid] - true_margin[valid]) ** 2))
            / margin_scale
        )
        if margin_scale > tie
        else float("nan")
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "pairwise_accuracy": pairwise,
        "weighted_pairwise_accuracy": weighted,
        "normalized_margin_rmse": normalized_margin_rmse,
        "pair_left": left,
        "pair_right": right,
        "true_margin": true_margin,
        "predicted_margin": predicted_margin,
        "pair_credit": credit,
        "pair_weight": weights,
    }

def bootstrap_mean(values, groups, repetitions, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    finite = np.isfinite(values)
    values = values[finite]
    groups = groups[finite]
    unique = np.unique(groups)
    if len(unique) == 0:
        return {
            "estimate": float("nan"),
            "low": float("nan"),
            "high": float("nan"),
            "n_clusters": 0,
        }
    grouped = [values[groups == group] for group in unique]
    rng = np.random.default_rng(seed)
    draws = np.empty(repetitions)
    for index in range(repetitions):
        sampled = rng.integers(0, len(unique), size=len(unique))
        draws[index] = np.mean(
            np.concatenate([grouped[item] for item in sampled])
        )
    return {
        "estimate": float(np.mean(values)),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "n_clusters": int(len(unique)),
        "n_bootstrap": int(repetitions),
    }

def load_truth_arrays():
    endpoints, costs, contacts = [], [], []
    for state_index in range(NUM_STATES):
        with np.load(TRUTH_DIR / f"state_{state_index:04d}.npz") as shard:
            endpoints.append(shard["endpoint_states"])
            costs.append(shard["physical_cost"])
            contacts.append(shard["contacts"])
    endpoints = np.asarray(endpoints, dtype=np.float64)
    return {
        "pose": pose_target(endpoints),
        "physical_cost": np.asarray(costs, dtype=np.float64),
        "contacts": np.asarray(contacts, dtype=np.int32),
    }

def load_model_arrays(model_name, projection_seed):
    with np.load(MODEL_ROOT / model_name / "state_0000.npz") as first:
        input_dim = int(first["predicted_features"].shape[-1])
    projection = random_projection(
        input_dim,
        READOUT_PROJECTION_DIM,
        projection_seed,
    )
    projected, latent_cost = [], []
    for state_index in range(NUM_STATES):
        with np.load(
            MODEL_ROOT / model_name / f"state_{state_index:04d}.npz"
        ) as shard:
            features = shard["predicted_features"].astype(np.float32)
            shape = features.shape[:-1]
            projected.append(
                (features.reshape(-1, input_dim) @ projection).reshape(
                    *shape, READOUT_PROJECTION_DIM
                )
            )
            latent_cost.append(shard["latent_predicted_cost"][0])
    return {
        "projected": np.asarray(projected, dtype=np.float32),
        "latent_cost": np.asarray(latent_cost, dtype=np.float64),
        "input_dim": input_dim,
        "projection_seed": projection_seed,
        "projection": projection,
    }

def flatten_states(values, state_indices):
    return np.asarray(values)[state_indices].reshape(
        -1, np.asarray(values).shape[-1]
    )

def fit_model_probes(model_name, model_arrays, truth, split, model_index):
    x = model_arrays["projected"]
    y = truth["pose"]
    x_train = flatten_states(x, split["train"])
    y_train = flatten_states(y, split["train"])
    x_calibration = flatten_states(x, split["calibration"])
    y_calibration = flatten_states(y, split["calibration"])

    linear = fit_linear_pose(
        x_train,
        y_train,
        x_calibration,
        y_calibration,
        RIDGE_LAMBDAS,
    )
    mlp = fit_mlp_pose(
        x_train,
        y_train,
        x_calibration,
        y_calibration,
        SEED + 1000 + model_index,
    )
    atomic_npz(
        PROBE_DIR / f"{model_name}_linear_pose.npz",
        mean=linear["mean"],
        scale=linear["scale"],
        coefficient=linear["coefficient"],
        ridge=np.asarray(linear["ridge"]),
    )
    atomic_npz(
        PROBE_DIR / f"{model_name}_projection.npz",
        matrix=model_arrays["projection"],
        seed=np.asarray(model_arrays["projection_seed"]),
    )
    checkpoint_path = PROBE_DIR / f"{model_name}_mlp_pose.pt"
    temporary = checkpoint_path.with_suffix(".tmp.pt")
    torch.save(
        {
            "state_dict": mlp["model"].state_dict(),
            "mean": mlp["mean"],
            "scale": mlp["scale"],
            "hidden": MLP_HIDDEN,
            "best_epoch": mlp["best_epoch"],
        },
        temporary,
    )
    temporary.replace(checkpoint_path)
    return linear, mlp

def evaluate_readouts(model_name, model_arrays, truth, split, linear, mlp):
    test_indices = split["test"]
    projected = model_arrays["projected"][test_indices]
    flat = projected.reshape(-1, projected.shape[-1])
    linear_pose = predict_linear_pose(linear, flat).reshape(
        projected.shape[:-1] + (4,)
    )
    mlp_pose = predict_mlp_pose(mlp, flat).reshape(
        projected.shape[:-1] + (4,)
    )
    true_pose = truth["pose"][test_indices]
    true_cost = truth["physical_cost"][test_indices]
    contacts = truth["contacts"][test_indices]
    latent_cost = model_arrays["latent_cost"][test_indices]
    linear_cost = task_cost_from_pose_target(linear_pose)
    mlp_cost = task_cost_from_pose_target(mlp_pose)
    readout_costs = {
        "latent_distance": latent_cost,
        "linear_pose": linear_cost,
        "mlp_pose": mlp_cost,
        "action_blind": np.zeros_like(true_cost),
        "linear_pose_shuffled": np.roll(linear_cost, 1, axis=1),
        "mlp_pose_shuffled": np.roll(mlp_cost, 1, axis=1),
        "oracle_pose": true_cost.copy(),
    }
    pose_predictions = {
        "linear_pose": linear_pose,
        "mlp_pose": mlp_pose,
    }

    unit_rows, pair_rows, action_rows = [], [], []
    for local_index, state_id in enumerate(test_indices):
        for horizon_index, horizon in enumerate(HORIZONS):
            truth_cost_vector = true_cost[local_index, :, horizon_index]
            truth_pose_vector = true_pose[local_index, :, horizon_index]
            contact_vector = contacts[local_index, :, horizon_index]
            for readout in READOUTS:
                prediction_cost_vector = readout_costs[readout][
                    local_index, :, horizon_index
                ]
                rank = ranking_metrics(
                    truth_cost_vector,
                    prediction_cost_vector,
                )
                if readout in pose_predictions:
                    predicted_pose_vector = pose_predictions[readout][
                        local_index, :, horizon_index
                    ]
                    pose_error = float(
                        np.mean(
                            physical_pose_error(
                                predicted_pose_vector,
                                truth_pose_vector,
                            )
                        )
                    )
                    physical_cost_rmse = float(
                        np.sqrt(
                            np.mean(
                                (
                                    prediction_cost_vector
                                    - truth_cost_vector
                                )
                                ** 2
                            )
                        )
                    )
                else:
                    predicted_pose_vector = None
                    pose_error = float("nan")
                    physical_cost_rmse = float("nan")
                unit_rows.append(
                    {
                        "state_id": int(state_id),
                        "model": model_name,
                        "readout": readout,
                        "horizon": int(horizon),
                        "pose_error": pose_error,
                        "physical_cost_rmse": physical_cost_rmse,
                        "top1_correct": rank["top1_correct"],
                        "regret": rank["regret"],
                        "normalized_regret": rank["normalized_regret"],
                        "pairwise_accuracy": rank["pairwise_accuracy"],
                        "weighted_pairwise_accuracy": rank[
                            "weighted_pairwise_accuracy"
                        ],
                        "normalized_margin_rmse": rank[
                            "normalized_margin_rmse"
                        ],
                        "contact_fraction": float(
                            np.mean(contact_vector > 0)
                        ),
                        "selected_action": rank["selected_action"],
                        "oracle_action": rank["oracle_action"],
                    }
                )
                for action in range(ACTIONS_PER_STATE):
                    row = {
                        "state_id": int(state_id),
                        "model": model_name,
                        "readout": readout,
                        "horizon": int(horizon),
                        "action": int(action),
                        "true_cost": float(truth_cost_vector[action]),
                        "predicted_cost": float(
                            prediction_cost_vector[action]
                        ),
                        "true_x": float("nan"),
                        "true_y": float("nan"),
                        "true_theta": float("nan"),
                        "predicted_x": float("nan"),
                        "predicted_y": float("nan"),
                        "predicted_theta": float("nan"),
                    }
                    if predicted_pose_vector is not None:
                        row.update(
                            {
                                "true_x": float(
                                    truth_pose_vector[action, 0]
                                ),
                                "true_y": float(
                                    truth_pose_vector[action, 1]
                                ),
                                "true_theta": float(
                                    angle_from_pose_target(
                                        truth_pose_vector[action]
                                    )
                                ),
                                "predicted_x": float(
                                    predicted_pose_vector[action, 0]
                                ),
                                "predicted_y": float(
                                    predicted_pose_vector[action, 1]
                                ),
                                "predicted_theta": float(
                                    angle_from_pose_target(
                                        predicted_pose_vector[action]
                                    )
                                ),
                            }
                        )
                    action_rows.append(row)
                for pair_index, (left, right) in enumerate(
                    zip(rank["pair_left"], rank["pair_right"])
                ):
                    contact_count = int(contact_vector[left] > 0) + int(
                        contact_vector[right] > 0
                    )
                    pair_rows.append(
                        {
                            "state_id": int(state_id),
                            "model": model_name,
                            "readout": readout,
                            "horizon": int(horizon),
                            "pair_left": int(left),
                            "pair_right": int(right),
                            "contact_stratum": [
                                "neither",
                                "one",
                                "both",
                            ][contact_count],
                            "true_margin": float(
                                rank["true_margin"][pair_index]
                            ),
                            "predicted_margin": float(
                                rank["predicted_margin"][pair_index]
                            ),
                            "ranking_credit": float(
                                rank["pair_credit"][pair_index]
                            ),
                            "margin_weight": float(
                                rank["pair_weight"][pair_index]
                            ),
                        }
                    )
    return unit_rows, pair_rows, action_rows

def summarize(unit_rows):
    metrics = [
        "pose_error",
        "physical_cost_rmse",
        "top1_correct",
        "normalized_regret",
        "pairwise_accuracy",
        "weighted_pairwise_accuracy",
        "normalized_margin_rmse",
    ]
    rows = []
    for model_name in MODEL_NAME:
        for readout in READOUTS:
            for horizon in HORIZONS:
                selected = [
                    row
                    for row in unit_rows
                    if row["model"] == model_name
                    and row["readout"] == readout
                    and row["horizon"] == horizon
                ]
                groups = np.asarray(
                    [row["state_id"] for row in selected]
                )
                summary = {
                    "model": model_name,
                    "readout": readout,
                    "horizon": int(horizon),
                    "num_test_states": int(len(selected)),
                }
                for metric_index, metric in enumerate(metrics):
                    interval = bootstrap_mean(
                        [row[metric] for row in selected],
                        groups,
                        BOOTSTRAP_REPS,
                        SEED
                        + 100 * MODEL_NAME.index(model_name)
                        + 10 * READOUTS.index(readout)
                        + metric_index
                        + horizon,
                    )
                    summary[metric] = interval["estimate"]
                    summary[f"{metric}_low"] = interval["low"]
                    summary[f"{metric}_high"] = interval["high"]
                rows.append(summary)
    return rows

def paired_improvement(unit_rows, readout, metric, higher_is_better):
    baseline = {
        (row["state_id"], row["model"], row["horizon"]): row[metric]
        for row in unit_rows
        if row["readout"] == "latent_distance"
    }
    candidate = {
        (row["state_id"], row["model"], row["horizon"]): row[metric]
        for row in unit_rows
        if row["readout"] == readout
    }
    keys = sorted(set(baseline) & set(candidate))
    if higher_is_better:
        values = np.asarray(
            [candidate[key] - baseline[key] for key in keys]
        )
    else:
        values = np.asarray(
            [baseline[key] - candidate[key] for key in keys]
        )
    groups = np.asarray([key[0] for key in keys])
    return bootstrap_mean(
        values,
        groups,
        BOOTSTRAP_REPS,
        SEED + 5000 + READOUTS.index(readout),
    )

def decision_payload(unit_rows):
    comparisons = {}
    for readout in ["linear_pose", "mlp_pose"]:
        comparisons[readout] = {
            "normalized_regret_improvement": paired_improvement(
                unit_rows,
                readout,
                "normalized_regret",
                higher_is_better=False,
            ),
            "weighted_pairwise_accuracy_improvement": paired_improvement(
                unit_rows,
                readout,
                "weighted_pairwise_accuracy",
                higher_is_better=True,
            ),
            "top1_accuracy_improvement": paired_improvement(
                unit_rows,
                readout,
                "top1_correct",
                higher_is_better=True,
            ),
        }
    linear_pass = (
        comparisons["linear_pose"]["normalized_regret_improvement"]["low"]
        > 0
        and comparisons["linear_pose"][
            "weighted_pairwise_accuracy_improvement"
        ]["low"]
        > 0
    )
    mlp_pass = (
        comparisons["mlp_pose"]["normalized_regret_improvement"]["low"]
        > 0
        and comparisons["mlp_pose"][
            "weighted_pairwise_accuracy_improvement"
        ]["low"]
        > 0
    )
    if linear_pass:
        status = "TASK_ALIGNED_SIGNAL"
    elif mlp_pass:
        status = "NONLINEAR_TASK_ALIGNED_SIGNAL"
    elif (
        comparisons["linear_pose"]["normalized_regret_improvement"]["high"]
        <= 0
        and comparisons["mlp_pose"]["normalized_regret_improvement"]["high"]
        <= 0
    ):
        status = "NO_TASK_ALIGNED_SIGNAL"
    else:
        status = "INCONCLUSIVE"
    return {
        "status": status,
        "primary_readout": "linear_pose",
        "baseline": "latent_distance",
        "primary_outcomes": [
            "physical_normalized_regret",
            "margin_weighted_pairwise_accuracy",
        ],
        "gate": (
            "state-clustered 95% bootstrap lower bounds above zero for "
            "both regret improvement and weighted pair-ranking improvement"
        ),
        "comparisons": comparisons,
        "interpretation_guardrail": (
            "This tests simulator planning with frozen world models and "
            "state-disjoint readouts; it does not establish real-robot reliability."
        ),
    }

def make_plots(summary_rows, action_rows):
    display_readouts = [
        "latent_distance",
        "linear_pose",
        "mlp_pose",
        "action_blind",
        "linear_pose_shuffled",
        "oracle_pose",
    ]
    labels = [
        "latent",
        "linear",
        "MLP",
        "blind",
        "shuffled",
        "oracle",
    ]
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(12, 4), squeeze=False)
    for column, horizon in enumerate(HORIZONS):
        values, lows, highs = [], [], []
        for readout in display_readouts:
            selected = [
                row
                for row in summary_rows
                if row["readout"] == readout
                and row["horizon"] == horizon
            ]
            values.append(float(np.mean([row["normalized_regret"] for row in selected])))
            lows.append(float(np.mean([row["normalized_regret_low"] for row in selected])))
            highs.append(float(np.mean([row["normalized_regret_high"] for row in selected])))
        errors = np.vstack(
            [np.asarray(values) - np.asarray(lows), np.asarray(highs) - np.asarray(values)]
        )
        axes[0, column].bar(np.arange(len(values)), values, yerr=errors, capsize=3)
        axes[0, column].set_xticks(np.arange(len(labels)), labels, rotation=30)
        axes[0, column].set_title(f"horizon {horizon}")
        axes[0, column].set_ylabel("physical normalized regret")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "regret_by_readout.png", dpi=180)
    plt.close(fig)

    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(12, 4), squeeze=False)
    for column, horizon in enumerate(HORIZONS):
        values = []
        for readout in display_readouts:
            selected = [
                row
                for row in summary_rows
                if row["readout"] == readout
                and row["horizon"] == horizon
            ]
            values.append(
                float(
                    np.mean(
                        [
                            row["weighted_pairwise_accuracy"]
                            for row in selected
                        ]
                    )
                )
            )
        axes[0, column].bar(np.arange(len(values)), values)
        axes[0, column].axhline(0.5, color="black", linewidth=1)
        axes[0, column].set_xticks(np.arange(len(labels)), labels, rotation=30)
        axes[0, column].set_ylim(0, 1)
        axes[0, column].set_title(f"horizon {horizon}")
        axes[0, column].set_ylabel("margin-weighted pair accuracy")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "weighted_ranking_by_readout.png", dpi=180)
    plt.close(fig)

    selected = [
        row
        for row in action_rows
        if row["readout"] in {"linear_pose", "mlp_pose"}
    ]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), squeeze=False)
    for column, readout in enumerate(["linear_pose", "mlp_pose"]):
        rows = [row for row in selected if row["readout"] == readout]
        truth = np.asarray([row["true_cost"] for row in rows])
        prediction = np.asarray([row["predicted_cost"] for row in rows])
        axes[0, column].scatter(truth, prediction, s=8, alpha=0.25)
        limit = max(float(np.max(truth)), float(np.max(prediction)))
        axes[0, column].plot([0, limit], [0, limit], color="black", linewidth=1)
        axes[0, column].set_title(readout)
        axes[0, column].set_xlabel("true physical cost")
        axes[0, column].set_ylabel("decoded physical cost")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / "decoded_vs_true_cost.png", dpi=180)
    plt.close(fig)

def checkpoint_manifest():
    candidates = []
    for root in [Path(os.environ["HF_HOME"]), Path(os.environ["TORCH_HOME"])]:
        if root.exists():
            for path in root.rglob("*"):
                if path.is_file() and (
                    "dino_wm_pusht" in path.name
                    or "jepa_wm_pusht" in path.name
                    or "dinov2_vits14" in path.name
                    or "dinov2_vits14" in str(path)
                ):
                    candidates.append(path)
    write_json(
        OUT / "checkpoints_manifest.json",
        {
            "models": MODEL_NAME,
            "repository": "facebook/jepa-wms",
            "encoder": "facebookresearch/dinov2:dinov2_vits14",
            "dataset_downloaded": False,
            "cached_files": [
                {
                    "path": str(path),
                    "size_bytes": path.stat().st_size,
                    "sha256": sha256_file(path),
                }
                for path in sorted(set(candidates))
            ],
        },
    )

def intermediate_manifest():
    files = [path for path in INTERMEDIATE.rglob("*") if path.is_file()]
    write_json(
        OUT / "intermediate_manifest.json",
        {
            "excluded_from_result_zip": True,
            "file_count": len(files),
            "total_bytes": sum(path.stat().st_size for path in files),
            "truth_shards": len(list(TRUTH_DIR.glob("state_*.npz"))),
            "model_shards": {
                model: len(list((MODEL_ROOT / model).glob("state_*.npz")))
                for model in MODEL_NAME
            },
        },
    )

def execute_analysis():
    split = make_state_split(NUM_STATES, PROBE_SPLIT, SEED + 17)
    write_json(
        OUT / "split_manifest.json",
        {
            "protocol": "state-disjoint train/calibration/final-test split",
            "train_states": split["train"],
            "calibration_states": split["calibration"],
            "test_states": split["test"],
        },
    )
    truth = load_truth_arrays()
    unit_rows, pair_rows, action_rows = [], [], []
    probe_manifest = {
        "world_models_frozen": True,
        "target": ["block_x_over_512", "block_y_over_512", "sin_theta", "cos_theta"],
        "projection_dim": READOUT_PROJECTION_DIM,
        "projection": "saved Gaussian matrix with entries N(0, 1/projection_dim)",
        "hyperparameter_selection": "calibration pose MSE only",
        "test_states_used_for_fitting": False,
        "models": {},
    }
    for model_index, model_name in enumerate(MODEL_NAME):
        projection_seed = SEED + 2000 + model_index
        arrays = load_model_arrays(model_name, projection_seed)
        linear, mlp = fit_model_probes(
            model_name,
            arrays,
            truth,
            split,
            model_index,
        )
        model_unit, model_pair, model_action = evaluate_readouts(
            model_name,
            arrays,
            truth,
            split,
            linear,
            mlp,
        )
        unit_rows.extend(model_unit)
        pair_rows.extend(model_pair)
        action_rows.extend(model_action)
        probe_manifest["models"][model_name] = {
            "raw_feature_dim": arrays["input_dim"],
            "projection_seed": arrays["projection_seed"],
            "linear_ridge": linear["ridge"],
            "linear_calibration_pose_mse": linear["calibration_pose_mse"],
            "mlp_hidden": MLP_HIDDEN,
            "mlp_best_epoch": mlp["best_epoch"],
            "mlp_calibration_pose_mse": mlp["calibration_pose_mse"],
        }
        del arrays, linear, mlp
        gc.collect()
        torch.cuda.empty_cache()
        gpu_report(f"{model_name}_probe_complete")

    write_csv(OUT / "unit_metrics.csv", unit_rows)
    write_csv(OUT / "pair_metrics.csv", pair_rows)
    write_csv(OUT / "action_predictions.csv", action_rows)
    summary_rows = summarize(unit_rows)
    write_csv(OUT / "metrics_summary.csv", summary_rows)
    decision = decision_payload(unit_rows)
    write_json(OUT / "stage2c_decision.json", decision)
    write_json(OUT / "probe_manifest.json", probe_manifest)
    write_json(
        OUT / "metrics_summary.json",
        {
            "status": "SUCCESS",
            "run_mode": RUN_MODE,
            "num_states": NUM_STATES,
            "num_test_states": len(split["test"]),
            "models": MODEL_NAME,
            "horizons": HORIZONS,
            "readouts": READOUTS,
            "decision": decision,
            "summary_rows": summary_rows,
        },
    )
    make_plots(summary_rows, action_rows)
    checkpoint_manifest()
    intermediate_manifest()
    if set(row["readout"] for row in unit_rows) != set(READOUTS):
        raise AssertionError("readout results are incomplete")
    if not all(
        np.isfinite(row["normalized_regret"])
        for row in unit_rows
    ):
        raise AssertionError("non-finite primary outcome")
    (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")
    gpu_report("analysis_complete")
    return decision

def package_results():
    result_zip = OUT.parent / "stage2c_result_bundle.zip"
    included = []
    with zipfile.ZipFile(
        result_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as archive:
        for path in sorted(OUT.rglob("*")):
            if not path.is_file():
                continue
            relative = path.relative_to(OUT)
            if relative.parts and relative.parts[0] == "intermediate":
                continue
            archive.write(path, arcname=str(relative))
            included.append(str(relative))
    write_json(
        OUT / "result_zip_manifest.json",
        {
            "archive": str(result_zip),
            "included_before_manifest_write": included,
            "intermediate_excluded": True,
        },
    )
    with zipfile.ZipFile(
        result_zip,
        "a",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as archive:
        archive.write(
            OUT / "result_zip_manifest.json",
            arcname="result_zip_manifest.json",
        )
    print(f"RESULT ZIP: {result_zip}")
    return result_zip

RUN_STATUS = "FAILED" if PIPELINE_FAILED else "PENDING"
if not PIPELINE_FAILED:
    try:
        STAGE2C_DECISION = execute_analysis()
        RUN_STATUS = "SUCCESS"
    except Exception:
        record_failure("task_aligned_analysis")
        RUN_STATUS = "FAILED"

RESULT_ZIP = package_results()
print("RUN_STATUS:", RUN_STATUS)
try:
    from google.colab import files

    files.download(str(RESULT_ZIP))
    print(f"Automatic download requested: {RESULT_ZIP.name}")
except Exception as download_exc:
    print("Automatic download unavailable; use the Colab Files pane.", download_exc)


In [ ]:
# Final compact status check. The preceding cell already requested the download.
if RUN_STATUS == "SUCCESS":
    required = [
        "config.json",
        "versions.json",
        "environment.json",
        "restore_test.json",
        "candidate_design_summary.json",
        "checkpoints_manifest.json",
        "split_manifest.json",
        "probe_manifest.json",
        "unit_metrics.csv",
        "pair_metrics.csv",
        "action_predictions.csv",
        "metrics_summary.csv",
        "metrics_summary.json",
        "stage2c_decision.json",
        "FAILURE_TRACE.txt",
        "logs/run.log",
        "plots/regret_by_readout.png",
        "plots/weighted_ranking_by_readout.png",
        "plots/decoded_vs_true_cost.png",
        "probes/dino_wm_pusht_linear_pose.npz",
        "probes/dino_wm_pusht_mlp_pose.pt",
        "probes/dino_wm_pusht_projection.npz",
        "probes/jepa_wm_pusht_linear_pose.npz",
        "probes/jepa_wm_pusht_mlp_pose.pt",
        "probes/jepa_wm_pusht_projection.npz",
    ]
    missing = [name for name in required if not (OUT / name).exists()]
    if missing:
        raise AssertionError(f"result bundle is missing: {missing}")
    assert (OUT / "FAILURE_TRACE.txt").read_text().strip() == "NONE"
    decision = json.loads((OUT / "stage2c_decision.json").read_text())
    print(json.dumps(decision, indent=2))
    print(f"Sanity checks passed. Return: {RESULT_ZIP.name}")
else:
    print("Stage 2C captured a failure.")
    print(f"Return {RESULT_ZIP.name}; it contains FAILURE_TRACE.txt and logs.")
